<a href="https://colab.research.google.com/github/abhigna-vemula/Movie-Script-Rating-Prediction/blob/main/Movie_Script_Rating_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

import pandas as pd
import numpy as np
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
dataset = load_dataset("IsmaelMousa/movies")
df = pd.DataFrame(dataset["train"])

df = df.rename(columns={"Name": "title", "Script": "script"})
df = df[["title", "script"]].dropna()

print("Loaded scripts:", len(df))
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

movies.json:   0%|          | 0.00/246M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Loaded scripts: 1172


,title,script
0,I am Sam,"""I AM SAM""\r\n\r\n ..."
1,"Book of Eli, The",THE BOOK OF ELI\r\n \r\n \r\...
2,"Sex, Lies and Videotape",""" s e x , l i e s , a n d v i d e o t a p e..."
3,"Passion of Joan of Arc, The",THE PASSION OF JOAN OF ARC\r\n\r\n\r\n\r\n ...
4,Wall Street,"""WALL STREET""\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\..."


In [ ]:
def calculate_rating(script):
    text = str(script)
    length = len(text.split())          # word count
    unique_words = len(set(text.split()))
    dialogue_count = text.count(":")

    # Weighted formula (Length + Vocabulary + Dialogue)
    score = (
        (length / 20000) * 4 +
        (unique_words / 3000) * 3 +
        (dialogue_count / 200) * 3
    )

    # Clamp values to [0,10]
    score = max(0, min(10, score))
    return round(score, 2)

df["rating"] = df["script"].apply(calculate_rating)
df[["title", "rating"]].head()

,title,rating
0,I am Sam,10.00
1,"Book of Eli, The",10.00
2,"Sex, Lies and Videotape",7.06
3,"Passion of Joan of Arc, The",10.00
4,Wall Street,10.00


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_script"] = df["script"].apply(clean_text)
df.head()

,title,script,rating,clean_script
0,I am Sam,"""I AM SAM""\r\n\r\n ...",10.00,i am sam screenplay by kristine johnson jessi...
1,"Book of Eli, The",THE BOOK OF ELI\r\n \r\n \r\...,10.00,the book of eli written by gary whitta june a ...
2,"Sex, Lies and Videotape",""" s e x , l i e s , a n d v i d e o t a p e...",7.06,s e x l i e s a n d v i d e o t a p e by stev...
3,"Passion of Joan of Arc, The",THE PASSION OF JOAN OF ARC\r\n\r\n\r\n\r\n ...,10.00,the passion of joan of arc written by carl the...
4,Wall Street,"""WALL STREET""\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\...",10.00,wall street wall street original screenplay b...


In [ ]:
print(df['clean_script'].iloc[0][:300])

 i am sam screenplay by kristine johnson jessie nelson shooting draft int starbucks a m we re watching a pair of hands arrange white sugar packets blue equal packets and pink sweet and low into small containers with precision and lightning speed the mixed up colors and crumpled packets are transform


In [ ]:
X = df["clean_script"]
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

len(X_train), len(X_test)

(937, 235)

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=50000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
model = LinearSVR(C=1.0)
model.fit(X_train_tfidf, y_train)

LinearSVR()

In [ ]:
y_pred = model.predict(X_test_tfidf)

MAE = mean_absolute_error(y_test, y_pred)
MSE = mean_squared_error(y_test, y_pred)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print("Regression Metrics:")
print("MAE :", round(MAE, 4))
print("MSE :", round(MSE, 4))
print("RMSE:", round(RMSE, 4))
print("R² Score:", round(R2, 4))

Regression Metrics:
MAE : 0.5614
MSE : 1.5201
RMSE: 1.2329
R² Score: 0.7875


In [ ]:
def predict_rating(script_text):
    cleaned = clean_text(script_text)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)[0]
    pred = max(0, min(10, pred))
    print("Predicted IMDb Rating:", round(pred, 2))

In [ ]:
# Test prediction
predict_rating("""
INT. DARK ROOM - NIGHT
A MAN sits alone at a table, staring at a broken photograph.
He takes a deep breath.
MAN: "This ends tonight."
""")

Predicted IMDb Rating: 9.94


In [ ]:
# Test prediction
predict_rating("""INT. DARK HOUSE - NIGHT
The house is quiet. Very quiet.
Shadows. Stillness. No movement.
""")

Predicted IMDb Rating: 9.65
